In [1]:
from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root

from custom_rules import apply_rules

import os
import ast

D:\projects\mldl\semevalpolar


In [2]:
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv')
gen = create_gen(data_path, batch_size=10, randomize=False)
generator_list = list(gen)

In [3]:
df = read_dataset(data_path)

In [4]:
predictions = []
usages = []

In [5]:
for batch in tqdm(generator_list[:10]):
    response = test_run(batch)
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 10/10 [01:10<00:00,  7.04s/it]


In [6]:
len(predictions)

10

In [7]:
flat = [x for sub in predictions for x in sub]

In [8]:
ground_truths = []

In [9]:
for i in range(len(flat) // len(predictions)):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [10]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, df["text"])
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

array([[72, 28],
       [ 0,  0]])

In [14]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong

,Predicted,Ground Truth,Text
0,1,0,is defending imperialism in the dnd chat
2,1,0,.senate.gov Theres 3 groups out there Republic...
4,1,0,"""bad people"" I have some conservative values s..."
5,1,0,"""Enemy of the people"" was a phrase coined and ..."
6,1,0,"""He also voiced support for Elon Musk, Tommy R..."
7,1,0,"""If you disagree with imperialism, youre a sup..."
9,1,0,"""No to ethnic cleansing"". QudsDay QudsMainIssue"
11,1,0,"""Saved Ukraine FOR NATO expansion"" you mean. R..."
13,1,0,"""This is another argument, if any were needed,..."
14,1,0,"""Well said. Multiculturalism enriches societie..."


In [18]:
submission = create_submission(df, flat)

In [19]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_urd.csv'), index=False)